In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import matplotlib.pyplot as plt
from aicsimageio import AICSImage
import platform
from scipy.ndimage import gaussian_filter


sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
from data_processing import *

system = platform.system()

if system == 'Linux':
    home = '/home/gerard/data/confocal/'
elif system == 'Darwin':
    home = '/Users/gerard/data/confocal/'
elif system == "Windows":
    home = 'C:/Users/cviko/data/confocal/'

11-Jun-26 13:55:17 - bfio.backends - WARNING  - Java backend is not available. This could be due to a missing dependency (jpype).
c:\Users\cviko\miniconda3\envs\leica-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
date = '2026_06_05'
user = 'Carmina'

info = describe_acquisition( home + date + '_' + user + '/Project.lif')

....

══════════════════════════════════════════════════════════════
Scene 0: dec02_5x_brain1alphaprime_L
══════════════════════════════════════════════════════════════
  Objective  : HC PL APO CS2    63x/1.40 OIL
  NA / n     : 1.4 / 1.518
  Voxel size : XY = 0.0226 µm/px  |  Z = 0.1307 µm/step
  Shape      : 61 × 512 × 512  (Z × Y × X)

  ────────────────────────────────────── Acquisition sequences
  Seq 1  laser(s): 488 nm + 638 nm  [simultaneous]
         pinhole  : 20.00 µm  (0.209 AU)
         det ch1  501–610 nm  Leica/ALEXA 488         em = 519 nm
         det ch2  643–788 nm  Leica/Cy5               em = 670 nm
  Seq 2  laser(s): 552 nm
         pinhole  : 20.00 µm  (0.209 AU)
         det ch2  557–761 nm  Leica/ALEXA 546         em = 573 nm

  ────────────────────────────────────── Image channels (0-indexed)
  ch0  Leica/ALEXA 488         em=519 nm  σ_xy=3.45px (0.078µm) [✓]  σ_z=2.03px [✓]  → 3D
  ch1  Leica/Cy5               em=670 nm  σ_xy=4.45px (0.101µm) [✓]  σ_z=2.62px 

In [4]:
loader2(date,user,False, False)

C:/Users/cviko/data/confocal/2026_06_05_Carmina
Number of series: 10
series 0: shape = (1, 3, 61, 512, 512)
series 1: shape = (1, 3, 134, 512, 512)
series 2: shape = (1, 3, 121, 512, 512)
series 3: shape = (1, 3, 88, 512, 512)
series 4: shape = (1, 3, 174, 512, 512)
series 5: shape = (1, 3, 121, 512, 512)
series 6: shape = (1, 3, 127, 512, 512)
series 7: shape = (1, 3, 197, 512, 512)
series 8: shape = (1, 3, 101, 512, 512)
series 9: shape = (1, 3, 94, 512, 512)


### DECONVOLUTION :
https://en.wikipedia.org/wiki/Richardson%E2%80%93Lucy_deconvolution

In [6]:


path_data = date + '_' + user + '/Project.lif'
path_data_all = home + path_data #os.path.join(hom

print(path_data_all)
img = AICSImage(path_data_all)
# print(len(img.scenes))
# print(img.scenes[0])



C:/Users/cviko/data/confocal/2026_06_05_Carmina/Project.lif


In [10]:
#list_numbers = [1,3,4,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
words = ['hola', 'adios', 'buenos dias', 'buenas tardes', 'buenas noches']
people = ['Alice', 'Bob', 'Charlie', 'David', 'Eve']
for name in people:
    for word in words:
        print(f'{name}, {word}')
    

Alice, hola
Alice, adios
Alice, buenos dias
Alice, buenas tardes
Alice, buenas noches
Bob, hola
Bob, adios
Bob, buenos dias
Bob, buenas tardes
Bob, buenas noches
Charlie, hola
Charlie, adios
Charlie, buenos dias
Charlie, buenas tardes
Charlie, buenas noches
David, hola
David, adios
David, buenos dias
David, buenas tardes
David, buenas noches
Eve, hola
Eve, adios
Eve, buenos dias
Eve, buenas tardes
Eve, buenas noches


In [11]:


series_list = list(range(len(info.keys())))

## DESIRED NUMBER OF ITERATIONS:
iterations = [
            # 1,        
            # 2,
            3,
            4,
            5,
            # 6,
            # 7, 
            #10, 
            # 15,
            20,#result[bg].std() / original[bg].std()
            ]

for series in series_list:
    img.set_scene(img.scenes[series])
    channels = list(range(len(info[list(info.keys())[series]]['image_channels'])))
    for channel in channels:   
        stack = img.get_image_data("ZYX", T=0, C=channel)
        for num_iter in iterations:
            print(f'iteration {num_iter} for series {series} channel {channel}')
            background_noise_ratios = []
            
            # actual deconvolution:
            result, sigma_xy_px = deconvolve(stack, path_data_all, channel=channel, scene=series, num_iter=num_iter)
            

iteration 3 for series 0 channel 0
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
2026_06_05_s0_ch0_deconv_iter_3.tif
iteration 4 for series 0 channel 0
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
2026_06_05_s0_ch0_deconv_iter_4.tif
iteration 5 for series 0 channel 0
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
2026_06_05_s0_ch0_deconv_iter_5.tif
iteration 20 for series 0 channel 0
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
2026_06_05_s0_ch0_deconv_iter_20.tif
iteration 3 for series 0 channel 1
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D
2026_06_05_s0_ch1_deconv_iter_3.tif
iteration 4 for series 0 channel 1
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D
2026_06_05_s0_ch1_deconv_iter_4.tif
iteration 5 for series 0 channel 1
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D
2026_

KeyboardInterrupt: 

In [1]:
import os
print(os.cpu_count())

18
